Reload the data in a new notebook

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q "/content/drive/MyDrive/projet_vision/kvasir-seg.zip" -d /content/data

import os, glob, random
import numpy as np
import cv2
import matplotlib.pyplot as plt

root = "/content/data"
img_dir = glob.glob(root + "/**/images", recursive=True)[0]
mask_dir = os.path.join(os.path.dirname(img_dir), "masks")

img_paths  = sorted(glob.glob(img_dir  + "/*.jpg"))
mask_paths = sorted(glob.glob(mask_dir + "/*.jpg"))

print(len(img_paths), len(mask_paths))
assert len(img_paths) == len(mask_paths)
assert all(os.path.basename(a) == os.path.basename(b)
           for a, b in zip(img_paths, mask_paths))
print("Images and masks match one to one")


from sklearn.model_selection import train_test_split

pairs = list(zip(img_paths, mask_paths))
train_pairs, temp_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
val_pairs, test_pairs   = train_test_split(temp_pairs, test_size=0.5, random_state=42)

print(len(train_pairs), len(val_pairs), len(test_pairs))


!pip install -q albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE = 256
MEAN = (0.485, 0.456, 0.406)   # ImageNet statistics, needed later for the pretrained ResNet
STD  = (0.229, 0.224, 0.225)

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])


import torch
from torch.utils.data import Dataset, DataLoader

class PolypDataset(Dataset):
    def __init__(self, pairs, transform):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        img_path, mask_path = self.pairs[i]
        img  = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)   # BGR -> RGB
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype("float32")                          # binarize: 0 or 1
        out = self.transform(image=img, mask=mask)
        return out["image"], out["mask"].unsqueeze(0)                  # mask -> (1, H, W)

BATCH = 16
train_loader = DataLoader(PolypDataset(train_pairs, train_tf), batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(PolypDataset(val_pairs,   val_tf),   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(PolypDataset(test_pairs,  val_tf),   batch_size=BATCH, shuffle=False, num_workers=2)



Mounted at /content/drive
1000 1000
Images and masks match one to one
800 100 100


the model

In [3]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        c = base
        self.pool = nn.MaxPool2d(2)
        # Encoder
        self.enc1 = DoubleConv(in_ch, c)
        self.enc2 = DoubleConv(c, c*2)
        self.enc3 = DoubleConv(c*2, c*4)
        self.enc4 = DoubleConv(c*4, c*8)
        # Bottleneck
        self.bott = DoubleConv(c*8, c*16)
        # Decoder
        self.up1 = nn.ConvTranspose2d(c*16, c*8, 2, stride=2); self.dec1 = DoubleConv(c*16, c*8)
        self.up2 = nn.ConvTranspose2d(c*8,  c*4, 2, stride=2); self.dec2 = DoubleConv(c*8,  c*4)
        self.up3 = nn.ConvTranspose2d(c*4,  c*2, 2, stride=2); self.dec3 = DoubleConv(c*4,  c*2)
        self.up4 = nn.ConvTranspose2d(c*2,  c,   2, stride=2); self.dec4 = DoubleConv(c*2,  c)
        # Output
        self.head = nn.Conv2d(c, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bott(self.pool(e4))
        d1 = self.dec1(torch.cat([self.up1(b),  e4], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d1), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d2), e2], dim=1))
        d4 = self.dec4(torch.cat([self.up4(d3), e1], dim=1))
        return self.head(d4)

test the shapes

In [4]:
device = "cuda"
model = UNet().to(device)

x = torch.randn(2, 3, 256, 256).to(device)
print("Output:", model(x).shape)       # expected: torch.Size([2, 1, 256, 256])
print("Parameters:", round(sum(p.numel() for p in model.parameters()) / 1e6, 2), "M")

Output: torch.Size([2, 1, 256, 256])
Parameters: 7.76 M


Loss functions and metrics

In [5]:
bce = nn.BCEWithLogitsLoss()

def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(1, 2, 3))
    total = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return 1 - ((2 * inter + eps) / (total + eps)).mean()

@torch.no_grad()
def metrics(logits, targets, thr=0.5, eps=1e-6):
    preds = (torch.sigmoid(logits) > thr).float()
    inter = (preds * targets).sum(dim=(1, 2, 3))
    p_sum = preds.sum(dim=(1, 2, 3))
    t_sum = targets.sum(dim=(1, 2, 3))
    iou  = ((inter + eps) / (p_sum + t_sum - inter + eps)).mean().item()
    dice = ((2 * inter + eps) / (p_sum + t_sum + eps)).mean().item()
    return iou, dice

Test on one batch with the untrained model

In [6]:
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)
out = model(x)
print("BCE      :", bce(out, y).item())
print("Dice loss:", dice_loss(out, y).item())
print("IoU, Dice:", metrics(out, y))

BCE      : 0.7053881883621216
Dice loss: 0.820018470287323
IoU, Dice: (0.10094517469406128, 0.17665249109268188)
